# FINN Montmeló MLP27K limpio

Notebook limpio para sustituir el flujo antiguo de `BobEsponja` por el modelo correcto del dataset de Montmeló.

**Idea:**

- Modelo bueno: `mlp27k_clean_inferred.onnx`
- Entrada del modelo limpio: `[1, 576]` con valores normalizados `0–1`
- Salida: `[1, 2]` con `vnorm, wnorm` en `0–1`
- Sin `TopK`, sin clasificación, sin 14 clases.

Este notebook primero valida el ONNX contra el `data.csv`, luego prepara un modelo con preprocesado `/255` para que el hardware pueda recibir bytes `UINT8`, y finalmente intenta lanzar el flujo FINN.

> Nota: si FINN falla en la conversión a HLS porque el ONNX es `float`, no significa que el modelo esté mal. Significa que hay que generar una versión cuantizada equivalente para FINN.


In [ ]:
import os
import sys
import csv
import math
import shutil
from pathlib import Path

import numpy as np
from PIL import Image

# Rutas típicas dentro del contenedor FINN
FINN_ROOT = Path('/home/miguel_tfg/FINN-v0.9-Stable')

# Añadimos rutas por si el kernel no las tiene ya en PYTHONPATH
for p in [
    FINN_ROOT / 'src',
    FINN_ROOT / 'deps/qonnx/src',
    FINN_ROOT / 'deps/finn-experimental/src',
    FINN_ROOT / 'deps/brevitas/src',
]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.core.datatype import DataType
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs, RemoveUnusedTensors

print('Imports OK')
print('Carpeta actual:', os.getcwd())

## 1. Configuración de rutas

Coloca `mlp27k_clean_inferred.onnx` en la misma carpeta que este notebook (`notebooks/UAH_example`).


In [ ]:
# ONNX bueno que hemos validado previamente
GOOD_ONNX = Path('mlp27k_clean_inferred.onnx')

# Nombres de trabajo
CLEAN_MODEL = Path('model_clean_mlp27k.onnx')
MODEL_FOR_FINN = Path('model_for_finn_mlp27k.onnx')
MODEL_WORK = Path('model.onnx')

# Dataset para validar contra CSV
DATASET_DIR_CANDIDATES = [
    Path('/tmp/Orig/DATA_12_16/counter_montmelo_holonomic'),
    Path('/tmp/Orig/Orig/DATA_12_16/counter_montmelo_holonomic'),
    Path('/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic'),
]

FINAL_OUTPUT_DIR = 'output_final_mlp27k'

assert GOOD_ONNX.exists(), f'No existe {GOOD_ONNX}. Copia aquí mlp27k_clean_inferred.onnx antes de seguir.'

shutil.copyfile(GOOD_ONNX, CLEAN_MODEL)
print('Copiado ONNX bueno:')
print(' ', GOOD_ONNX, '->', CLEAN_MODEL)

for d in DATASET_DIR_CANDIDATES:
    if (d / 'data.csv').exists():
        DATASET_DIR = d
        break
else:
    raise FileNotFoundError('No encuentro DATA_12_16/counter_montmelo_holonomic/data.csv')

CSV_PATH = DATASET_DIR / 'data.csv'
print('Dataset:', DATASET_DIR)
print('CSV:', CSV_PATH)

## 2. Inspección del ONNX limpio

Debe salir:

- Input `[1, 576]`
- Output `[1, 2]`
- Nodos tipo `Gemm`/`Relu`


In [ ]:
model = ModelWrapper(str(CLEAN_MODEL))
model = model.transform(InferShapes())
model.save(str(CLEAN_MODEL))

print('INPUTS:')
for inp in model.graph.input:
    print(' ', inp.name, model.get_tensor_shape(inp.name))

print('
OUTPUTS:')
for out in model.graph.output:
    print(' ', out.name, model.get_tensor_shape(out.name))

print('
NODOS:')
for node in model.graph.node:
    print(' ', node.name, node.op_type)

## 3. Validación del ONNX limpio contra el CSV

Aquí el modelo recibe entrada `0–1`. Esta prueba debe dar algo parecido a:

- `MAE_v ≈ 0.60`
- `MAE_w ≈ 0.13`

Si esto sale parecido, el ONNX es el bueno.


In [ ]:
N_SAMPLES = 2000


def image_to_flat_0_1(img_path):
    img = Image.open(img_path).convert('RGB')
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    # HWC -> CHW -> RRR GGG BBB
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)
    arr = arr / 255.0
    return arr.reshape(1, 576).astype(np.float32)


def denorm_v(vnorm):
    return 13.6 * vnorm - 0.6


def denorm_w(wnorm):
    return 2.0 * math.pi * wnorm - math.pi


def raw_from_norm_0_1(x):
    return int(round(max(0.0, min(1.0, x)) * 255.0))

model = ModelWrapper(str(CLEAN_MODEL))
model = model.transform(InferShapes())
input_name = model.graph.input[0].name
output_name = model.graph.output[0].name

err_v, err_w = [], []
pred_v, pred_w = [], []

n = 0
with open(CSV_PATH) as f:
    reader = csv.DictReader(f)
    for row in reader:
        img_path = DATASET_DIR / row['image_name']
        if not img_path.exists():
            continue

        x = image_to_flat_0_1(img_path)
        y = execute_onnx(model, {input_name: x})[output_name].reshape(-1)

        vnorm = float(y[0])
        wnorm = float(y[1])
        v_pred = denorm_v(vnorm)
        w_pred = denorm_w(wnorm)

        v_csv = float(row['v'])
        w_csv = float(row['w'])

        err_v.append(v_pred - v_csv)
        err_w.append(w_pred - w_csv)
        pred_v.append(v_pred)
        pred_w.append(w_pred)

        if n < 10:
            raw_v_csv = raw_from_norm_0_1((v_csv + 0.6) / 13.6)
            raw_w_csv = raw_from_norm_0_1((w_csv + math.pi) / (2.0 * math.pi))
            raw_v_pred = raw_from_norm_0_1(vnorm)
            raw_w_pred = raw_from_norm_0_1(wnorm)
            print(f"{row['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} raw_csv={raw_v_csv},{raw_w_csv} | ONNX v={v_pred:.4f} w={w_pred:.4f} raw_onnx={raw_v_pred},{raw_w_pred}")

        n += 1
        if n >= N_SAMPLES:
            break

err_v = np.asarray(err_v)
err_w = np.asarray(err_w)
pred_v = np.asarray(pred_v)
pred_w = np.asarray(pred_w)

print('
Frames:', n)
print('MAE_v =', np.mean(np.abs(err_v)))
print('MAE_w =', np.mean(np.abs(err_w)))
print('v pred min/max/std =', pred_v.min(), pred_v.max(), pred_v.std())
print('w pred min/max/std =', pred_w.min(), pred_w.max(), pred_w.std())

## 4. Preparar modelo para FINN con entrada `UINT8`

El ONNX limpio espera entrada `0–1`. Para que el hardware pueda recibir bytes `0–255`, añadimos un preprocesado equivalente a dividir entre 255.

**Importante:** no se añade `TopK`, porque la salida debe seguir siendo `[vnorm, wnorm]`.


In [ ]:
# Intentamos crear model_for_finn_mlp27k.onnx con preprocesado /255.
# Si falla por alguna incompatibilidad de versiones, se puede seguir trabajando con CLEAN_MODEL como referencia.

try:
    import brevitas.onnx as bo
    from finn.util.pytorch import ToTensor
    from qonnx.transformation.merge_onnx_models import MergeONNXModels

    model = ModelWrapper(str(CLEAN_MODEL))
    model = model.transform(InferShapes())
    global_inp_name = model.graph.input[0].name
    ishape = model.get_tensor_shape(global_inp_name)
    print('Shape del modelo limpio:', ishape)

    totensor_pyt = ToTensor()
    PREPROC_ONNX = Path('preproc_div255_mlp27k.onnx')
    bo.export_finn_onnx(totensor_pyt, ishape, str(PREPROC_ONNX))

    pre_model = ModelWrapper(str(PREPROC_ONNX))
    merged = model.transform(MergeONNXModels(pre_model))

    # La entrada global ahora son bytes UINT8
    global_inp_name = merged.graph.input[0].name
    merged.set_tensor_datatype(global_inp_name, DataType['UINT8'])

    # Tidy-up básico
    merged = merged.transform(InferShapes())
    merged = merged.transform(FoldConstants())
    merged = merged.transform(GiveUniqueNodeNames())
    merged = merged.transform(GiveReadableTensorNames())
    merged = merged.transform(InferDataTypes())
    merged = merged.transform(RemoveStaticGraphInputs())
    merged = merged.transform(RemoveUnusedTensors())

    merged.save(str(MODEL_FOR_FINN))
    shutil.copyfile(MODEL_FOR_FINN, MODEL_WORK)

    print('Modelo para FINN guardado en:', MODEL_FOR_FINN)
    print('También copiado como:', MODEL_WORK)

except Exception as e:
    print('No se pudo crear el modelo con preprocesado UINT8.')
    print('Error:', repr(e))
    print('Copio el modelo limpio como model.onnx para inspección, pero recuerda que espera entrada 0-1.')
    shutil.copyfile(CLEAN_MODEL, MODEL_WORK)

## 5. Inspección del modelo que se intentará pasar a FINN

Comprueba que no aparece `TopK` ni `LabelSelect`. La salida debe seguir teniendo 2 valores.


In [ ]:
model = ModelWrapper(str(MODEL_WORK))
model = model.transform(InferShapes())
model.save(str(MODEL_WORK))

print('Modelo de trabajo:', MODEL_WORK)

print('
INPUTS:')
for inp in model.graph.input:
    print(' ', inp.name, model.get_tensor_shape(inp.name), model.get_tensor_datatype(inp.name))

print('
OUTPUTS:')
for out in model.graph.output:
    print(' ', out.name, model.get_tensor_shape(out.name), model.get_tensor_datatype(out.name))

print('
Nodos:')
for node in model.graph.node:
    print(' ', node.name, node.op_type)

bad_nodes = [n.op_type for n in model.graph.node if n.op_type in ['TopK', 'LabelSelect']]
assert len(bad_nodes) == 0, f'Hay nodos de clasificación que no deberían estar: {bad_nodes}'
print('
OK: no hay TopK ni LabelSelect')

## 6. Intento de flujo FINN automático

Este paso intenta generar reports y stitched IP.

Puede fallar si FINN no acepta el modelo por ser `float`. Si falla aquí, la conclusión no es que el modelo esté mal, sino que hay que generar una versión cuantizada FINN del MLP27K.


In [ ]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

DELETE_PREVIOUS_OUTPUT = True

if DELETE_PREVIOUS_OUTPUT and os.path.exists(FINAL_OUTPUT_DIR):
    shutil.rmtree(FINAL_OUTPUT_DIR)
    print('Borrado output anterior:', FINAL_OUTPUT_DIR)

cfg = build_cfg.DataflowBuildConfig(
    output_dir=FINAL_OUTPUT_DIR,
    fpga_part='xc7z020clg400-1',
    synth_clk_period_ns=10.0,
    mvau_wwidth_max=36,
    save_intermediate_models=True,
    stitched_ip_gen_dcp=True,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
    ]
)

print('Lanzando build_dataflow con:', MODEL_WORK)
print('Output dir:', FINAL_OUTPUT_DIR)

try:
    build.build_dataflow_cfg(str(MODEL_WORK), cfg)
    print('FINN build terminado')
except Exception as e:
    print('
FINN build ha fallado.')
    print('Error:', repr(e))
    print('
Interpretación: si el error habla de Gemm/MatMul/float/no quantization, hay que cuantizar este MLP antes de generar hardware.')
    raise

## 7. Comprobaciones de salida

Si el build llega hasta el final, aquí deberían aparecer los ficheros de `stitched_ip`.


In [ ]:
out = Path(FINAL_OUTPUT_DIR)
print('Existe output:', out.exists(), out)

if out.exists():
    print('
Contenido output:')
    for p in sorted(out.iterdir()):
        print(' ', p)

    stitched = out / 'stitched_ip'
    if stitched.exists():
        print('
Contenido stitched_ip:')
        for p in sorted(stitched.iterdir()):
            print(' ', p.name)

    report = out / 'report'
    if report.exists():
        print('
Contenido report:')
        for p in sorted(report.iterdir()):
            print(' ', p.name)

# Resumen técnico

Este notebook elimina los elementos que venían del flujo antiguo de clasificación:

- No usa `BobEsponja`
- No usa `state_dict_quantized_last.pth`
- No usa entrada `(1,1,64,64)`
- No usa `TopK`
- No usa `LabelSelect`

En su lugar usa:

- `mlp27k_clean_inferred.onnx`
- Entrada `576 = 16×12×3`
- Salida `2 = v,w`
- Normalización de entrada `0–1`
- Normalización de salida `0–1`

Si FINN no acepta el modelo, el siguiente paso es cuantizar esta misma arquitectura `576 → 32 → 256 → 2` y exportarla otra vez como ONNX compatible con FINN.
